In [8]:
!pip install yfinance
import yfinance as yf
from pathlib import Path

data_dir = Path('../data/raw')

# Download SPY daily data up to today
spy = yf.download("SPY", start="1990-01-01", auto_adjust=False, progress=False)

# Save to CSV
output_path = data_dir / "SPY_daily_yahoo_raw.csv"
spy.to_csv(output_path)

print(f"Data saved to: {output_path}")
print(f"Latest date in data: {spy.index[-1].date()}")
print(f"Rows: {len(spy)}")

Data saved to: ..\data\raw\SPY_daily_yahoo_raw.csv
Latest date in data: 2026-05-14
Rows: 8380


In [9]:
import numpy as np
import pandas as pd
import xarray as xr

# Load data
df = pd.read_csv(
    '../data/raw/SPY_daily_yahoo_raw.csv',
    skiprows=[0,1,2], header=None,
    names=['Date','Adj Close','Close','High','Low','Open','Volume'],
    index_col=0, parse_dates=True
)

df = df.sort_index()
P = df['Adj Close'].values
dates = df.index

# Best parameters from your research
tau = 200
q = 0.9

# Compute threshold from training data (up to end 2024)
P_train = df[df.index <= '2024-12-31']['Adj Close'].values
inc_train = P_train[tau:] - P_train[:-tau]
thresh = np.quantile(inc_train, q)

# Today's signal
inc_now = P[-1] - P[-1 - tau]
signal = inc_now >= thresh

print(f"Current 200-day increment: {inc_now:.2f}")
print(f"Threshold (90th percentile): {thresh:.2f}")
print(f"Signal: {'BUY SPY' if signal else 'NO TRADE'}")
print(f"Today's date: {dates[-1].date()}")

Current 200-day increment: 118.25
Threshold (90th percentile): 37.42
Signal: BUY SPY
Today's date: 2026-05-14
